# YOLOv8n ONNX：三类边缘部署图改造实验

这份 notebook 用同一份 YOLOv8n ONNX，分别演示：

1. **算子兼容与改写**：把模型中实际出现的 2 倍最近邻 Resize 改写为分组 ConvTranspose。实验假设目标编译器不支持该 Resize 配置、但支持 ConvTranspose；这只是兼容性演示，实际设备必须先查支持表。
2. **冗余图清理**：原始 YOLOv8n 没有 Identity 或明确的死分支。为避免误称模型本身有冗余，本实验在一条真实计算路径上插入 Identity，并添加一条无用分支，再安全清理它们。
3. **Shape／Layout 适配**：原模型接收 NCHW，给模型加一个 NHWC 输入边界和 Transpose，保持内部卷积图不变，并对比推理结果。

所有变体都从原始模型单独复制，不会串联前一项实验的改动。模型、权重和实验产物放在仓库根目录的 .weights 中，该目录已被 .gitignore 忽略。

数值对比使用 ONNX Runtime CPU。它能验证改写前后的 ONNX 输出是否一致，但不能代替目标 NPU 的编译、运行和精度测试。

In [1]:
from pathlib import Path
import shutil
import urllib.request

import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, numpy_helper
from collections import Counter

def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / ".git").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("请从 Yunshao 仓库目录或其子目录启动 notebook。")

ROOT = find_repo_root()
MODEL_DIR = ROOT / ".weights" / "models" / "yolov8n-demo"
MODEL_PATH = MODEL_DIR / "yolov8n.onnx"
PT_PATH = MODEL_DIR / "yolov8n.pt"
RUN_DIR = MODEL_DIR / "notebook_experiments"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# .weights 在本机被 Git 忽略。克隆仓库后若模型不存在，本单元会下载权重并导出固定尺寸 ONNX。
if not MODEL_PATH.exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    if not PT_PATH.exists():
        url = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt"
        print("下载 YOLOv8n 权重：", url)
        urllib.request.urlretrieve(url, PT_PATH)
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise RuntimeError("导出缺失的 ONNX 需要 Ultralytics；请先运行 %pip install ultralytics") from exc
    YOLO(str(PT_PATH)).export(
        format="onnx", imgsz=640, batch=1, dynamic=False, simplify=False, opset=17
    )

model = onnx.load(str(MODEL_PATH))
onnx.checker.check_model(model)
print("模型：", MODEL_PATH)
print("ONNX opset：", [(op.domain or "ai.onnx", op.version) for op in model.opset_import])
print("节点数：", len(model.graph.node))

模型： d:\workplace\Yunshao\.weights\models\yolov8n-demo\yolov8n.onnx
ONNX opset： [('ai.onnx', 17)]
节点数： 263


## 基线：先认识模型接口和图

模型输入为 NCHW 的固定尺寸张量，输出是检测头原始预测；这个导出的图没有 NonMaxSuppression，后处理留在模型外。两处 Resize 都是最近邻放大 2 倍，正好可以做一个精确、可测的算子改写实验。

随机输入用于检查变换是否保持数值计算，不用于评估检测精度。真实图片推理还需匹配 Ultralytics 的颜色顺序、letterbox、归一化和后处理配置。

In [2]:
op_counts = Counter(node.op_type for node in model.graph.node)
input_info = model.graph.input[0]
output_info = model.graph.output[0]

def tensor_shape(value_info):
    return [
        dim.dim_value if dim.dim_value else (dim.dim_param or "?")
        for dim in value_info.type.tensor_type.shape.dim
    ]

print("输入：", input_info.name, tensor_shape(input_info))
print("输出：", output_info.name, tensor_shape(output_info))
print("算子计数：", dict(op_counts))
print("NMS 节点数：", sum(node.op_type == "NonMaxSuppression" for node in model.graph.node))
print("Identity 节点数：", sum(node.op_type == "Identity" for node in model.graph.node))

input_name = input_info.name
input_shape = tuple(tensor_shape(input_info))
assert input_shape == (1, 3, 640, 640), f"此实验预期固定 NCHW 输入，实际为 {input_shape}"

rng = np.random.default_rng(2026)
x_nchw = rng.random(input_shape, dtype=np.float32)

def make_session(path):
    return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])

baseline_session = make_session(MODEL_PATH)
baseline_outputs = baseline_session.run(None, {input_name: x_nchw})
print("基线推理输出形状：", [array.shape for array in baseline_outputs])

def compare_with_baseline(candidate_path, candidate_input, candidate_input_name="images",
                          rtol=1e-5, atol=1e-5):
    candidate_session = make_session(candidate_path)
    candidate_outputs = candidate_session.run(
        None, {candidate_input_name: candidate_input}
    )
    assert len(candidate_outputs) == len(baseline_outputs)
    diffs = [
        float(np.max(np.abs(reference - candidate)))
        for reference, candidate in zip(baseline_outputs, candidate_outputs)
    ]
    np.testing.assert_allclose(
        candidate_outputs, baseline_outputs, rtol=rtol, atol=atol
    )
    print("逐输出最大绝对误差：", diffs)
    print("通过：候选模型输出在设定容差内匹配基线。")
    return diffs

输入： images [1, 3, 640, 640]
输出： output0 [1, 84, 8400]
算子计数： {'Conv': 64, 'Sigmoid': 58, 'Mul': 60, 'Constant': 24, 'Split': 8, 'Add': 9, 'Concat': 17, 'MaxPool': 3, 'Resize': 2, 'Reshape': 8, 'Transpose': 1, 'Softmax': 1, 'Shape': 1, 'Gather': 1, 'Div': 2, 'Slice': 2, 'Sub': 2}
NMS 节点数： 0
Identity 节点数： 0
基线推理输出形状： [(1, 84, 8400)]


## 实验一：算子兼容与改写

### 目标

假设目标芯片的编译器不支持 YOLOv8n 图中的 Resize 最近邻配置，但支持分组 ConvTranspose。这里利用 Resize 的精确参数：scale 为 [1, 1, 2, 2]，coordinate_transformation_mode 为 asymmetric，nearest_mode 为 floor。对这种 2 倍最近邻插值，每个输入像素会复制到输出的 2×2 区域。

用每个通道独立的 ConvTranspose：权重形状 [C,1,2,2] 且全为 1，group=C，kernel=2×2，stride=2。因为卷积核与步长相同，各输入像素恰好复制到四个输出位置，不会发生重叠累加。

这不是对任意 Resize 的通用替换。线性插值、其他坐标规则、非整数 scale、动态 sizes 都不满足此推导。ONNX 对 Resize 的坐标规则和 ConvTranspose 的分组/输出尺寸有明确语义，替换前必须逐项核对。

In [3]:
def constant_value(graph_model, tensor_name):
    for initializer in graph_model.graph.initializer:
        if initializer.name == tensor_name:
            return numpy_helper.to_array(initializer)
    for node in graph_model.graph.node:
        if node.op_type == "Constant" and tensor_name in node.output:
            for attribute in node.attribute:
                if attribute.name == "value":
                    return numpy_helper.to_array(attribute.t)
    raise KeyError(f"找不到常量张量：{tensor_name}")

def attr_map(node):
    result = {}
    for attribute in node.attribute:
        value = helper.get_attribute_value(attribute)
        if isinstance(value, bytes):
            value = value.decode("utf-8")
        result[attribute.name] = value
    return result

def rewrite_resize_2x_as_grouped_convtranspose(source_model):
    rewritten = onnx.shape_inference.infer_shapes(source_model)
    shape_by_name = {}
    for value_info in (
        list(rewritten.graph.input)
        + list(rewritten.graph.value_info)
        + list(rewritten.graph.output)
    ):
        shape_by_name[value_info.name] = tensor_shape(value_info)

    new_nodes = []
    new_weights = []
    records = []
    for node in rewritten.graph.node:
        if node.op_type != "Resize":
            new_nodes.append(node)
            continue

        attributes = attr_map(node)
        scales = constant_value(rewritten, node.input[2]) if len(node.input) > 2 and node.input[2] else None
        source_shape = shape_by_name.get(node.input[0])
        matches = (
            scales is not None
            and np.array_equal(scales.astype(np.float32), np.array([1, 1, 2, 2], dtype=np.float32))
            and attributes.get("mode") == "nearest"
            and attributes.get("coordinate_transformation_mode") == "asymmetric"
            and attributes.get("nearest_mode") == "floor"
            and source_shape is not None
            and len(source_shape) == 4
            and isinstance(source_shape[1], int)
        )
        if not matches:
            new_nodes.append(node)
            continue

        channels = source_shape[1]
        weight_name = f"demo_resize_as_deconv_weights_{len(records)}"
        new_weights.append(
            numpy_helper.from_array(
                np.ones((channels, 1, 2, 2), dtype=np.float32),
                name=weight_name,
            )
        )
        new_nodes.append(
            helper.make_node(
                "ConvTranspose",
                inputs=[node.input[0], weight_name],
                outputs=list(node.output),
                kernel_shape=[2, 2],
                strides=[2, 2],
                group=channels,
                name=(node.name or "Resize") + "_as_grouped_ConvTranspose",
            )
        )
        records.append({
            "replaced_node": node.name,
            "input_shape": source_shape,
            "channels/group": channels,
            "scales": scales.tolist(),
        })

    if not records:
        raise RuntimeError("没有找到满足本实验严格前提的 Resize 节点。")
    del rewritten.graph.node[:]
    rewritten.graph.node.extend(new_nodes)
    rewritten.graph.initializer.extend(new_weights)
    rewritten = onnx.shape_inference.infer_shapes(rewritten)
    onnx.checker.check_model(rewritten)
    return rewritten, records

resize_model, resize_records = rewrite_resize_2x_as_grouped_convtranspose(model)
resize_path = RUN_DIR / "01_resize_to_grouped_convtranspose.onnx"
onnx.save(resize_model, str(resize_path))
print("改写记录：")
for record in resize_records:
    print(record)
print("改写后 Resize 数：", sum(n.op_type == "Resize" for n in resize_model.graph.node))
print("改写后 ConvTranspose 数：", sum(n.op_type == "ConvTranspose" for n in resize_model.graph.node))
resize_error = compare_with_baseline(resize_path, x_nchw)

改写记录：
{'replaced_node': '/model.10/Resize', 'input_shape': [1, 256, 20, 20], 'channels/group': 256, 'scales': [1.0, 1.0, 2.0, 2.0]}
{'replaced_node': '/model.13/Resize', 'input_shape': [1, 128, 40, 40], 'channels/group': 128, 'scales': [1.0, 1.0, 2.0, 2.0]}
改写后 Resize 数： 0
改写后 ConvTranspose 数： 2
逐输出最大绝对误差： [0.0]
通过：候选模型输出在设定容差内匹配基线。


### 实验一结论

数值误差若为 0 或处于容差内，说明在这个 YOLOv8n、固定输入和这两组 Resize 参数下，改写保持了 ONNX CPU 的输出。

部署时再查目标编译器是否支持分组 ConvTranspose、权重布局、kernel/stride 属性，以及该节点的数据类型和形状。若目标反而不支持 ConvTranspose，这种改写就没有价值；应寻找目标支持的另一种等价计算或把插值放到模型外。

## 实验二：冗余图清理

### 目标

原始 YOLOv8n 没有 Identity 节点，也没有可直接指出的孤立死节点，因此我们不把真实节点误判成冗余。这里在一条真实 Conv→下游节点的边上插入 Identity，再添加一个不连接模型输出的 Add→Relu 死分支。

清理分两步：

1. Identity 消除：把 Identity 输出的所有消费者改接到 Identity 输入，然后删除 Identity。
2. 死代码消除：从模型输出反向追踪依赖，只保留能影响图输出的节点。

下面的死代码清理器只处理主图，不递归处理控制流子图或本地函数；真实通用工具必须递归处理这些图结构或使用成熟的图优化器。

In [4]:
def insert_redundancies(source_model):
    result = onnx.ModelProto()
    result.CopyFrom(source_model)
    graph_nodes = list(result.graph.node)
    graph_outputs = {value.name for value in result.graph.output}

    selected = None
    for producer_index, producer in enumerate(graph_nodes):
        if producer.op_type != "Conv" or not producer.output:
            continue
        tensor_name = producer.output[0]
        if tensor_name in graph_outputs:
            continue
        consumers = [
            (node_index, input_index)
            for node_index, consumer in enumerate(graph_nodes)
            for input_index, name in enumerate(consumer.input)
            if name == tensor_name
        ]
        if len(consumers) == 1:
            selected = producer_index, tensor_name, consumers[0]
            break
    if selected is None:
        raise RuntimeError("没找到只有一个消费者的 Conv 输出边。")

    producer_index, tensor_name, (consumer_index, input_index) = selected
    identity_output = "demo_identity_output"
    graph_nodes[consumer_index].input[input_index] = identity_output
    identity = helper.make_node(
        "Identity", [tensor_name], [identity_output],
        name="demo_inserted_identity",
    )
    graph_nodes.insert(producer_index + 1, identity)

    # 这条支路的结果不连接任何模型输出，专供死代码清理实验。
    graph_nodes.extend([
        helper.make_node(
            "Add", [tensor_name, tensor_name], ["demo_dead_add_output"],
            name="demo_dead_add",
        ),
        helper.make_node(
            "Relu", ["demo_dead_add_output"], ["demo_dead_relu_output"],
            name="demo_dead_relu",
        ),
    ])
    del result.graph.node[:]
    result.graph.node.extend(graph_nodes)
    onnx.checker.check_model(result)
    return result, tensor_name

def remove_named_identity(graph_model, identity_name):
    nodes = list(graph_model.graph.node)
    identity = next(
        node for node in nodes
        if node.op_type == "Identity" and node.name == identity_name
    )
    source_name = identity.input[0]
    identity_output = identity.output[0]
    if identity_output in {output.name for output in graph_model.graph.output}:
        raise ValueError("本演示不删除直接作为模型输出的 Identity。")
    for node in nodes:
        if node is identity:
            continue
        for index, value_name in enumerate(node.input):
            if value_name == identity_output:
                node.input[index] = source_name
    nodes = [node for node in nodes if node is not identity]
    del graph_model.graph.node[:]
    graph_model.graph.node.extend(nodes)

def prune_top_level_dead_nodes(graph_model):
    nodes = list(graph_model.graph.node)
    producer_by_value = {
        output_name: index
        for index, node in enumerate(nodes)
        for output_name in node.output
        if output_name
    }
    needed_values = [output.name for output in graph_model.graph.output]
    live_nodes = set()
    while needed_values:
        value_name = needed_values.pop()
        producer_index = producer_by_value.get(value_name)
        if producer_index is None or producer_index in live_nodes:
            continue
        live_nodes.add(producer_index)
        needed_values.extend(
            name for name in nodes[producer_index].input if name
        )
    kept = [node for index, node in enumerate(nodes) if index in live_nodes]
    removed_names = [
        node.name for index, node in enumerate(nodes) if index not in live_nodes
    ]
    del graph_model.graph.node[:]
    graph_model.graph.node.extend(kept)
    return removed_names

redundant_model, chosen_edge = insert_redundancies(model)
redundant_path = RUN_DIR / "02_with_deliberate_redundancies.onnx"
onnx.save(redundant_model, str(redundant_path))
print("插入 Identity 的真实边：", chosen_edge)
print("基线节点数：", len(model.graph.node))
print("插入后节点数：", len(redundant_model.graph.node))
print("插入后 Identity 数：", sum(n.op_type == "Identity" for n in redundant_model.graph.node))
redundant_error = compare_with_baseline(redundant_path, x_nchw)

cleaned_model = onnx.ModelProto()
cleaned_model.CopyFrom(redundant_model)
remove_named_identity(cleaned_model, "demo_inserted_identity")
removed_dead = prune_top_level_dead_nodes(cleaned_model)
cleaned_model = onnx.shape_inference.infer_shapes(cleaned_model)
onnx.checker.check_model(cleaned_model)
cleaned_path = RUN_DIR / "02_cleaned.onnx"
onnx.save(cleaned_model, str(cleaned_path))
print("死分支清理掉的节点：", removed_dead)
print("清理后节点数：", len(cleaned_model.graph.node))
print("清理后 Identity 数：", sum(n.op_type == "Identity" for n in cleaned_model.graph.node))
cleaned_error = compare_with_baseline(cleaned_path, x_nchw)

插入 Identity 的真实边： /model.22/cv2.0/cv2.0.2/Conv_output_0
基线节点数： 263
插入后节点数： 266
插入后 Identity 数： 1
逐输出最大绝对误差： [0.0]
通过：候选模型输出在设定容差内匹配基线。
死分支清理掉的节点： ['demo_dead_add', 'demo_dead_relu']
清理后节点数： 263
清理后 Identity 数： 0
逐输出最大绝对误差： [0.0]
通过：候选模型输出在设定容差内匹配基线。


### 实验二结论

这个可控实验中，清理前后的输出应匹配基线；Identity 和 Add→Relu 死分支被移除。对真实模型做清理时，先证明节点输出不能沿任何路径影响模型输出。Netron 中 Constant 没有单独画成方框，不代表它无用或已从 ONNX 删除。

ONNX Runtime 也能执行基础图优化（例如常量折叠和冗余节点消除）；它适合作为参照，但应检查生成模型、节点变化和目标平台表现，不要把减少节点数当作唯一目标。

## 实验三：Shape／Layout 适配

### 目标

基线输入是 [1,3,640,640]，即 NCHW。现在构造一个新输入契约 [1,640,640,3]，在图入口加 Transpose，使用 perm=[0,3,1,2] 将 NHWC 转回 NCHW，然后接回原 YOLOv8n 主干。

这演示的是**边界适配**：内部 Conv 图不变，输入张量数据和输入 shape 同步改变。仅修改 shape 元数据不会重排数据。ONNX Conv 的逻辑输入仍是 [N,C,H,W]，所以不能因为硬件内部采用 channels-last 就擅自改写整张图。

当前 YOLOv8n 是静态 640×640 输入。Shape Inference 可帮助检查图中的 Shape/Reshape 关系，但不会自动让固定尺寸模型获得任意动态尺寸支持。Resize 等节点和检测头 reshape 也必须能适配目标尺寸。

In [5]:
nhwc_model = onnx.ModelProto()
nhwc_model.CopyFrom(model)
external_name = nhwc_model.graph.input[0].name
internal_name = "demo_images_nchw"

original_nodes = list(nhwc_model.graph.node)
rewired_input_count = 0
for node in original_nodes:
    for index, value_name in enumerate(node.input):
        if value_name == external_name:
            node.input[index] = internal_name
            rewired_input_count += 1
if rewired_input_count == 0:
    raise RuntimeError("没有找到原始图像输入的消费者。")

nhwc_transpose = helper.make_node(
    "Transpose",
    inputs=[external_name],
    outputs=[internal_name],
    perm=[0, 3, 1, 2],
    name="demo_nhwc_to_nchw",
)
del nhwc_model.graph.node[:]
nhwc_model.graph.node.extend([nhwc_transpose] + original_nodes)

input_dims = nhwc_model.graph.input[0].type.tensor_type.shape.dim
for dim, value in zip(input_dims, [1, 640, 640, 3]):
    dim.ClearField("dim_param")
    dim.dim_value = value

nhwc_model = onnx.shape_inference.infer_shapes(nhwc_model)
onnx.checker.check_model(nhwc_model)
nhwc_path = RUN_DIR / "03_nhwc_input_adapter.onnx"
onnx.save(nhwc_model, str(nhwc_path))

x_nhwc = np.transpose(x_nchw, (0, 2, 3, 1)).copy()
print("新输入契约：", nhwc_model.graph.input[0].name, tensor_shape(nhwc_model.graph.input[0]))
print("入口 Transpose：", [list(n.attribute[0].ints) for n in nhwc_model.graph.node[:1]])
print("原图像输入被接入 Transpose 的消费者数：", rewired_input_count)
nhwc_error = compare_with_baseline(nhwc_path, x_nhwc, candidate_input_name=external_name)

新输入契约： images [1, 640, 640, 3]
入口 Transpose： [[0, 3, 1, 2]]
原图像输入被接入 Transpose 的消费者数： 1
逐输出最大绝对误差： [0.0]
通过：候选模型输出在设定容差内匹配基线。


## 总结与迁移到真实设备

| 场景 | 本实验改动 | 必须满足的前提 | 通过标准 |
|---|---|---|---|
| 算子兼容 | Resize → 分组 ConvTranspose | 仅限 scale=[1,1,2,2] 的 asymmetric/floor nearest；目标确实支持替代算子 | ONNX 检查通过，CPU 输出与基线匹配，目标编译/推理通过 |
| 冗余图清理 | 移除插入的 Identity 与死分支 | 删除后节点输出不影响任何模型输出；清理范围覆盖相应子图 | 清理后节点回到基线数量，输出匹配 |
| Shape/Layout | NHWC 输入 + 边界 Transpose | 输入形状与实际内存布局一致，内部仍得到原 NCHW 张量 | 输入/输出 shape 正确，输出匹配 |

实验产物位于 .weights/models/yolov8n-demo/notebook_experiments/，不会进入 Git。当前模型导出不含 NMS；若部署需要 NMS 而 NPU 不支持，常见方案是保持 YOLO 原始预测输出，在 CPU/应用层实现后处理，并核对框格式、类别策略和阈值。

### 参考

- [ONNX Resize](https://onnx.ai/onnx/operators/onnx__Resize.html)
- [ONNX ConvTranspose](https://onnx.ai/onnx/operators/onnx__ConvTranspose.html)
- [ONNX Transpose](https://onnx.ai/onnx/operators/onnx__Transpose.html)
- [ONNX Shape Inference](https://onnx.ai/onnx/api/shape_inference.html)
- [ONNX Model Checker](https://onnx.ai/onnx/api/checker.html)
- [ONNX Runtime Graph Optimizations](https://onnxruntime.ai/docs/performance/model-optimizations/graph-optimizations.html)

In [6]:
experiment_summary = [
    ("Resize → grouped ConvTranspose", resize_error),
    ("Identity + dead-branch cleanup", cleaned_error),
    ("NHWC input adapter", nhwc_error),
]
print("所有场景均与 YOLOv8n NCHW 基线完成数值比较：")
for name, errors in experiment_summary:
    print(f"- {name}: max_abs_error={max(errors):.8g}")

所有场景均与 YOLOv8n NCHW 基线完成数值比较：
- Resize → grouped ConvTranspose: max_abs_error=0
- Identity + dead-branch cleanup: max_abs_error=0
- NHWC input adapter: max_abs_error=0
